In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import random
import gc
import sys
import pprint
import scrublet as scr

In [2]:
# --- Set a global random seed for reproducibility ---
seed = 97
random.seed(seed)
np.random.seed(seed)

print(f"Global random seed set to: {seed}")

# --- Configure plotting parameters ---
sc.set_figure_params(frameon=False, fontsize=10, dpi=100)
# This section demonstrates how to add a custom module path.
# The 'anno_self' module contains user-defined annotation functions.
# Ensure this path points to the directory containing your custom scripts.
try:
    sys.path.append(r"/home/lishaogang/NEU/validation-set")
    import anno_self
    print("Custom annotation module 'anno_self' loaded successfully.")
except ImportError:
    print("Warning: Custom annotation module 'anno_self' not found. Annotation steps will require manual implementation.")

In [3]:
os.chdir("/home/lishaogang/NEU2026-Jan")

In [7]:
adata_T = sc.read_h5ad('adata_T_raw.h5ad')

In [ ]:
import scanpy as sc
import numpy as np

# 1. 过滤掉 T 细胞数量太少的样本（样本内至少要有 20 个 T 细胞才参与 HVG 计算）
min_cells_for_hvg = 20
cell_counts = adata_T.obs['orig.ident'].value_counts()
valid_samples = cell_counts[cell_counts >= min_cells_for_hvg].index



# 剔除那些只有零星几个 T 细胞的样本
adata_T = adata_T[adata_T.obs['orig.ident'].isin(valid_samples)].copy()
print(f"--- 过滤后剩余样本数: {len(valid_samples)} ---")

print(f"--- 过滤后剩余 T 细胞数: {adata_T.n_obs} ---")


In [16]:
# 2. 重新过滤基因（只保留在当前 T 细胞子集中有表达的基因）
sc.pp.filter_genes(adata_T, min_cells=3)
# 3. 运行 HVG
print("\n--- Finding HVGs within the T-cell subset ---")
# 方案 A: 依然使用 batch_key（如果样本量大且细胞多，推荐）
try:
    sc.pp.highly_variable_genes(adata_T, batch_key='dataset', n_top_genes=2000, flavor='seurat')
except Exception as e:
    print(f"依然报错: {e}。自动切换到方案 B：全局 HVG 计算...")
    # 方案 B: 如果还是报错，说明样本间差异过大或仍有极小样本，则不带 batch_key 运行
    sc.pp.highly_variable_genes(adata_T, n_top_genes=2000, flavor='seurat')

# 4. 提取 HVG 并后续处理
adata_T_hvg = adata_T[:, adata_T.var.highly_variable].copy()
sc.pp.scale(adata_T_hvg, zero_center=False)
sc.tl.pca(adata_T_hvg, n_comps=50, random_state=seed)

print("✅ PCA 计算完成！")

In [17]:
# --- Drawing a Scree Plot ---
print("Drawing a scree plot to assist in determining the selection of n_pcs...")

# 1. Modify Matplotlib's global default figure size before plotting
#    This will affect all figures created afterwards
plt.rcParams['figure.figsize'] = (8, 6)  # Set to 8x6 inches

# 2. Now directly call scanpy's plotting function without any additional parameters
#    When it internally creates the figure, it will follow the global size set above
sc.pl.pca_variance_ratio(adata_T_hvg, log=True, n_pcs=50)
plt.show()

# 3. Restore Matplotlib's default settings
#    This is a good practice to prevent this size setting from affecting your subsequent plots
plt.rcParams.update(plt.rcParamsDefault)

In [18]:

print("\n--- Re-running Harmony on the T-cell subset ---")
sc.external.pp.harmony_integrate(
    adata_T_hvg,
    key='orig.ident',
    basis='X_pca',
    adjusted_basis='X_pca_harmony',
    random_state=seed
)

adata_T.obsm['X_pca'] = adata_T_hvg.obsm['X_pca']
adata_T.obsm['X_pca_harmony'] = adata_T_hvg.obsm['X_pca_harmony']

In [18]:
print("\n--- Re-calculating UMAP and sub-clustering ---")
sc.pp.neighbors(adata_T,n_neighbors = 25, n_pcs=30, use_rep='X_pca_harmony', random_state=seed)
sc.tl.umap(adata_T, random_state=seed)

In [19]:
sc.tl.leiden(adata_T, resolution=2.0, key_added='leiden_T', random_state=seed)

In [20]:
dir()

In [23]:
# --- 7. 可视化 ---
sc.pl.umap(adata_T, color=['leiden_T', 'dataset'])

In [24]:
sc.pl.umap(adata_T, color='cell_subtype')

In [25]:
# 1. 定义你想要的完整基因列表
genes_requested = [
    'CD3D', 'CD4', 'CD8A', 'CD8B', 'NCAM1', 'NCR1', 'TRABD2A', 'MAL', 
    'CCR7', 'SELL', 'LEF1', 'TCF7', 'GPR183', 'CD44', 'S100A4', 'CTLA4', 
    'IL2RA', 'IL7R', 'CX3CR1', 'GZMA', 'GZMH', 'NKG7', 'GNLY', 'MKI67', 
    'CABP4', 'NOTCH1', 'TRDC', 'TRDV1', 'TRDV2', 'SLC4A10', 'KLRB1', 
    'KLRF1', 'KLRC2', 'KLRC3', 'TYROBP', 'FCGR3A',    'NKG7',  'FGFBP2', # NK
    'KLRB1','TRAC','TRBC1','TRBC2','HBB',"PPBP"] # NKT
    

# 2. 检查哪些基因在数据中真正可用
# 如果 use_raw=True，检查 adata_T.raw.var_names；否则检查 adata_T.var_names
if adata_T.raw is not None:
    available_genes = adata_T.raw.var_names
else:
    available_genes = adata_T.var_names

genes_to_plot = [gene for gene in genes_requested if gene in available_genes]

# 3. 打印出哪些基因被剔除了，方便你知晓数据情况
missing_genes = set(genes_requested) - set(genes_to_plot)
if missing_genes:
    print(f"以下基因在数据中未找到，已跳过: {missing_genes}")

# 4. 运行绘图
dotplot_T = sc.pl.dotplot(
    adata_T,
    genes_to_plot,
    groupby='leiden_T',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    figsize=(16, 6), # 基因较多，建议调宽 figsize
    return_fig=True,
    show=True
)

In [26]:
sc.pl.dotplot(
    adata_T,
    genes_to_plot,
    groupby='leiden_T',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    figsize=(12, 6),
    show=True
)

In [86]:
sc.pl.dotplot(
    adata_T_clean,
    ['SELL','CCR7','GZMA','NKG7','GNLY'],
    groupby='leiden_T',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    figsize=(12, 6),
    show=True
)

In [29]:
import pandas as pd
import importlib
importlib.reload(anno_self) # 强制刷新修改后的脚本

print("\n--- 正在导出点图的原始数据 ---")

# 1. 获取数据并处理
mean_expression_df = dotplot_T.dot_color_df.copy()
fraction_df = dotplot_T.dot_size_df.copy()

# 2. 检查是否有重复列名（如果是重复基因，这里会显示）
if mean_expression_df.columns.duplicated().any():
    print("⚠️ 发现重复的基因名，已通过 iloc 逻辑自动处理。")

# 3. 赋予唯一的数字索引，防止 .loc 赋值冲突
original_names = mean_expression_df.index.tolist()
unique_idx = pd.Index(range(len(mean_expression_df)))
mean_expression_df.index = unique_idx
fraction_df.index = unique_idx

# 4. 调用函数
try:
    # 生成标记表 (+, ++, low等)
    final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)
    # 生成最终描述文字
    summaries = anno_self.generate_cluster_summaries(final_summary_df)
    
    # 5. 还原名称
    summaries.index = original_names

    # 6. 完整打印
    with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
        for i, (cluster_name, description) in enumerate(summaries.iMemorys()):
            print(f"[{i}] {cluster_name} :\n{description}\n")
            print("-" * 80)
            
except Exception as e:
    print(f"❌ 运行失败: {e}")

In [28]:
import importlib
importlib.reload(anno_self)

In [32]:
adata_T.write_h5ad("adata_T_final0.h5ad")

In [4]:
adata_T = sc.read_h5ad("adata_T_final0.h5ad")

In [31]:
sc.pl.umap(adata_T, color=['leiden_T'], legend_loc = 'on data')

In [75]:
sc.pl.umap(
    adata_T_clean, 
    color='leiden_T', 
    groups=['24'],          # 只给第27簇上色
    legend_loc='on data',   # 标签显示在簇上
    na_color='lightgray',   # 非选中簇的颜色，默认为很浅的灰色
    palette='Set1'          # 可选：指定选中簇的颜色盘
)

In [70]:
annotation_dict = {
    "0": "CD4 Memory",
    "1": "CD4 Naive",
    "2": "CD8 CTL",
    "3": "CD4 Memory",
    "4": "CD4 Memory",
    "5": "CD4 Memory",
    "6": "CD8 Naive",
    "7": "CD4 Memory",
    "8": "CD8 CTL",
    "9": "CD4 CTL",
    "10": "CD4 Memory",
    "11": "CD4 Treg",
    "12": "MAIT",
    "13": "CD4 CTL",
    "14": "CD8 Memory",
    "15": "CD4 Memory",
    "16": "CD8 CTL",
    "17": "CD8 Memory",
    "18": "NKT",
    "19": "γδT 2",
    "20": "Low Quality/Immature",
    "21": "Low Quality/Immature",
    "22": "CD8 CTL",
    "23": "Low Quality/Immature",
    "24": "CD4 Naive",
    "25": "RBC Contaminated T",
    "26": "CD4 Memory",
    "27": "CD4 Naive",
    "28": "CD4 Memory",
    "29": "CD4 Memory",
    "30": "RBC Contaminated T",
    "31": "CD4 Memory",
    "32": "CD4 Memory",
    "33": "CD4 Naive",
    "34": "CD8 CTL",
    "35": "CD4 Naive",
    "36": "γδT 2",
    "37": "RBC Contaminated T"
}

In [45]:
adata_T.obs['cell_subtype'] = (
    adata_T.obs['leiden_T'].astype(str).map(annotation_dict).astype('category')
)

In [46]:
from cProfile import label


sc.pl.umap(
    adata_T, 
    color=['cell_subtype'],
    cmap='Reds',          # 使用高对比度的红色系
    vmax='p99',           # 将颜色上限设为99%分位数，增强对比
    ncols=4              # 将3个基因图排成一行
)
sc.pl.umap(
    adata_T, 
    color=['leiden_T'],
    cmap='Reds',          # 使用高对比度的红色系
    vmax='p99',           # 将颜色上限设为99%分位数，增强对比
    ncols=4 ,
    legend_loc='on data'             # 将3个基因图排成一行
)

In [49]:
# 1. 检查质量指标：基因数和线粒体比例
sc.pl.violin(adata_T, ['n_genes_by_counts', 'pct_counts_mt'], groupby='leiden_T', order=['27', '28'])

# 2. 检查是否有特定的非 T 细胞污染（如红细胞或髓系信号）
sc.pl.umap(adata_T, color=['HBB', 'LYZ', 'CD14'], groups=['27', '28'])

# 3. 统计细胞数量
print(adata_T.obs['leiden_T'].value_counts()[['27', '28']])

In [ ]:
sc.tl.rank_genes_groups(
    adata_T, 
    groupby='leiden_T', 
    groups=['29'], 
    reference='rest', 
    method='t-test'
)


In [ ]:
# 提取 27 簇和 28 簇的前 5 个 Marker 基因
n_genes = 10
marker_genes_27 = adata_T.uns['rank_genes_groups']['names']['27'][:n_genes].tolist()
marker_genes_28 = adata_T.uns['rank_genes_groups']['names']['28'][:n_genes].tolist()


# 构造一个字典，用于在 Dotplot 中对基因进行分组显示
var_names = {
    'Cluster 27 Markers': marker_genes_27,
    'Cluster 28 Markers': marker_genes_28
}

In [66]:
marker_genes_29 = adata_T.uns['rank_genes_groups']['names']['29'][:15].tolist()

In [67]:
sc.pl.dotplot(
    adata_T, 
    var_names=marker_genes_29,       # 刚才提取的基因字典
    groupby='leiden_T',        # 显示所有簇
    standard_scale='var',      # 归一化，方便跨簇观察特异性
    cmap='Reds'
)

In [ ]:
#删除adata_T 中

In [71]:
# 1. 定义需要删除的污染簇和低质量簇 ID
# 20, 21, 23: 低质量/未成熟/特征模糊
# 25, 30: 红细胞污染 (HBB+)
# 27, 32: 髓系/中性粒细胞污染 (S100A8/A9+)
# 28: 肥大细胞污染 (MS4A2+)
# 37: 严重死细胞/环境 RNA 污染 (MALAT1++, HBB+)
clusters_to_remove = ['20', '21', '23', '25', '27', '28', '30', '32', '37','29']

# 2. 执行过滤操作
# 使用 ~ 符号表示“取反”，即保留不在列表中的簇
adata_T_clean = adata_T[~adata_T.obs['leiden_T'].isin(clusters_to_remove)].copy()

# 3. 非常重要：清除掉已删除簇的分类标签（Category）
# 如果不执行这一步，后续绘图的图例中依然会出现这些空簇的名字
adata_T_clean.obs['leiden_T'] = adata_T_clean.obs['leiden_T'].cat.remove_unused_categories()

# 4. 打印清理前后的细胞数量对比
print(f"清理前细胞数: {adata_T.n_obs}")
print(f"清理后细胞数: {adata_T_clean.n_obs}")
print(f"共剔除了 {adata_T.n_obs - adata_T_clean.n_obs} 个低质量细胞。")



In [58]:
from cProfile import label


sc.pl.umap(
    adata_T_clean, 
    color=['cell_subtype'],
    cmap='Reds',          # 使用高对比度的红色系
    vmax='p99',           # 将颜色上限设为99%分位数，增强对比
    ncols=4              # 将3个基因图排成一行
)
sc.pl.umap(
    adata_T_clean, 
    color=['leiden_T'],
    cmap='Reds',          # 使用高对比度的红色系
    vmax='p99',           # 将颜色上限设为99%分位数，增强对比
    ncols=4 ,
    legend_loc='on data'             # 将3个基因图排成一行
)

In [72]:
sc.pl.umap(
    adata_T_clean, 
    color=['leiden_T']
    
    #legend_loc='on data'             # 将3个基因图排成一行
)

In [79]:
# 2. 强制对齐旧的 Harmony 坐标 (防止报错)
# 这一步是为了不重跑 Harmony 也能直接画图
adata_T_clean.obsm['X_pca_harmony'] = adata_T[adata_T_clean.obs_names, :].obsm['X_pca_harmony']




In [80]:
# 3. 重新计算 UMAP（删除了污染簇后，UMAP 布局会更清晰）
sc.pp.neighbors(adata_T_clean, n_neighbors=25, n_pcs=30, use_rep='X_pca_harmony')



In [82]:
sc.tl.umap(adata_T_clean)

In [87]:

sc.pl.umap(adata_T_clean, color=['leiden_T','cell_subtype'], legend_loc='on data')

In [88]:
# 3. 重新运行 Leiden（建议换一个名字，如 'leiden_clean'，以便和旧的对比）
# 调整 resolution 得到你满意的精细度
sc.tl.leiden(adata_T_clean, resolution=1.0, key_added='leiden_clean')
# 4. 绘图观察

In [95]:
# 重新运行Leiden聚类（适配未来版本，无警告）
sc.tl.leiden(
    adata_T_clean,
    resolution=2.0,
    key_added='leiden_clean_2.0',
    flavor="igraph",       # 未来默认后端（替代leidenalg）
    n_iterations=2,        # 未来默认迭代次数
    directed=False         # igraph后端要求directed=False
)

In [96]:
sc.pl.umap(adata_T_clean, color=['leiden_clean_2.0'], legend_loc='on data')

In [98]:
# 1. 定义你想要的完整基因列表
genes_requested = [
    'CD3D', 'CD4', 'CD8A', 'CD8B', 'NCAM1', 'NCR1', 'TRABD2A', 'MAL', 
    'CCR7', 'SELL', 'LEF1', 'TCF7', 'GPR183', 'CD44', 'S100A4', 'CTLA4', 
    'IL2RA', 'IL7R', 'CX3CR1', 'GZMA', 'GZMH', 'NKG7', 'GNLY', 'MKI67', 
    'CABP4', 'NOTCH1', 'TRDC', 'TRDV1', 'TRDV2', 'SLC4A10', 'KLRB1', 
    'KLRF1', 'KLRC2', 'KLRC3', 'TYROBP', 'FCGR3A',    'NKG7',  'FGFBP2', # NK
    'KLRB1','TRAC','TRBC1','TRBC2','HBB',"PPBP"] # NKT
    

# 2. 检查哪些基因在数据中真正可用
# 如果 use_raw=True，检查 adata_T.raw.var_names；否则检查 adata_T.var_names
if adata_T.raw is not None:
    available_genes = adata_T.raw.var_names
else:
    available_genes = adata_T.var_names

genes_to_plot = [gene for gene in genes_requested if gene in available_genes]

# 3. 打印出哪些基因被剔除了，方便你知晓数据情况
missing_genes = set(genes_requested) - set(genes_to_plot)
if missing_genes:
    print(f"以下基因在数据中未找到，已跳过: {missing_genes}")

# 4. 运行绘图
dotplot_T = sc.pl.dotplot(
    adata_T_clean,
    genes_to_plot,
    groupby='leiden_clean_2.0',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    figsize=(16, 6), # 基因较多，建议调宽 figsize
    return_fig=True,
    show=True
)

In [101]:
sc.pl.dotplot(
    adata_T_clean,
    genes_to_plot,
    groupby='leiden_clean_2.0',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    figsize=(12, 6),
    show=True
)

In [102]:
import pandas as pd
import importlib
importlib.reload(anno_self) # 强制刷新修改后的脚本

print("\n--- 正在导出点图的原始数据 ---")

# 1. 获取数据并处理
mean_expression_df = dotplot_T.dot_color_df.copy()
fraction_df = dotplot_T.dot_size_df.copy()

# 2. 检查是否有重复列名（如果是重复基因，这里会显示）
if mean_expression_df.columns.duplicated().any():
    print("⚠️ 发现重复的基因名，已通过 iloc 逻辑自动处理。")

# 3. 赋予唯一的数字索引，防止 .loc 赋值冲突
original_names = mean_expression_df.index.tolist()
unique_idx = pd.Index(range(len(mean_expression_df)))
mean_expression_df.index = unique_idx
fraction_df.index = unique_idx

# 4. 调用函数
try:
    # 生成标记表 (+, ++, low等)
    final_summary_df = anno_self.generate_summary_table(mean_expression_df, fraction_df)
    # 生成最终描述文字
    summaries = anno_self.generate_cluster_summaries(final_summary_df)
    
    # 5. 还原名称
    summaries.index = original_names

    # 6. 完整打印
    with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
        for i, (cluster_name, description) in enumerate(summaries.iMemorys()):
            print(f"[{i}] {cluster_name} :\n{description}\n")
            print("-" * 80)
            
except Exception as e:
    print(f"❌ 运行失败: {e}")

In [115]:
cell_type_annotation = {
    '0': 'CD4 Naïve',
    '1': 'CD4 Memory',
    '2': 'CD4 Memory',
    '3': 'CD4 Memory',
    '4': 'CD4 Memory',
    '5': 'CD4 Memory',
    '6': 'CD4 Memory',
    '7': 'CD4 Memory',
    '8': 'CD4 Memory',
    '9': 'CD8 Naïve',
    '10': 'CD4 Memory',
    '11': 'CD4 Treg',
    '12': 'CD8 Memory',
    '13': 'CD8 CTL',
    '14': 'CD8 Memory',
    '15': 'CD8 Memory',
    '16': 'NKT',
    '17': 'CD8 Memory',
    '18': 'γδT',
    '19': 'CD4 CTL',
    '20': 'CD4 Memory',
    '21': 'MAIT',
    '22': 'CD4 Memory',
    '23': 'CD4 Memory',
    '24': 'CD4 Memory',
    '25': 'γδT'
}

In [116]:
adata_T_clean.obs['cell_subtype1'] = (
    adata_T_clean.obs['leiden_clean_2.0'].astype(str).map(cell_type_annotation).astype('category')
)

In [120]:
sc.pl.umap(adata_T_clean, color=['cell_subtype','cell_subtype1','leiden_clean_2.0'], legend_loc='on data')

In [118]:
sc.pl.umap(adata_T_clean, color=['cell_subtype','cell_subtype1',]
           #, legend_loc='on data'
          )

In [123]:
adata_T_clean.obs['cell_subtype'] = adata_T_clean.obs['cell_subtype1'].copy()

In [124]:
adata_T_clean.write_h5ad("adata_T_final.h5ad")

In [125]:
genes_to_plot

In [126]:
sc.pl.dotplot(
    adata_T_clean,
    genes_to_plot,
    groupby='cell_subtype',
    use_raw=True,
    standard_scale='var',
    cmap='Reds',
    figsize=(12, 6),
    show=True
)

In [139]:
# 定义细胞类型的逻辑顺序
cell_subtype_order = [
     'CD4 Treg', 'CD4 Naïve', 'CD4 Memory','CD4 CTL', 
    'CD8 Naïve', 'CD8 Memory', 'CD8 CTL', 
    'MAIT', 'γδT', 'NKT'
]

# 将 adata 中的类别设置为有序，这样画图时 Y 轴会按此顺序排列
adata_T_clean.obs['cell_subtype'] = adata_T_clean.obs['cell_subtype'].astype('category')
adata_T_clean.obs['cell_subtype'] = adata_T_clean.obs['cell_subtype'].cat.reorder_categories(cell_subtype_order)

In [140]:
# 按照阶梯状效果重新排列基因
genes_ladder = [
    # --- 公共/谱系标志 ---
    'CD3D', 'TRAC', 'TRBC1', 
    
    # --- CD4 & Naive 组 ---
    'CD4',  # --- CD8 谱系 ---
    'CD8A', 'CD8B',
    'IL2RA', 'CTLA4',
    
    'TRABD2A', 'MAL', 'CCR7', 'SELL', 'LEF1', 'TCF7', 
    
    # --- Memory & Treg 组 ---
    'GPR183', 'CD44', 'S100A4', 
    
   
    
    # --- 细胞毒性 (CTL) 组 ---
    'GZMA', 'GZMH', 'NKG7', 'GNLY', 'CX3CR1', 'FGFBP2',
    
    # --- 特殊亚群 (MAIT / γδT / NKT) ---
    'SLC4A10', 'KLRB1',          # MAIT 核心
    'TRDC', 'TRDV1', 'TRDV2',    # γδT 核心
    'NCAM1', 'NCR1', 'KLRF1', 'KLRC2', 'KLRC3', 'TYROBP', 'FCGR3A' # NKT 核心
]

# 移除重复的基因名（如果有的话）
genes_ladder = list(dict.fromkeys(genes_ladder))

In [141]:
import scanpy as sc

# # 定义 Marker 分组（可选，可以让图上方出现分组色块，更清晰）
# var_groups = {
#     'Naive': ['CCR7', 'SELL', 'LEF1', 'TCF7'],
#     'Memory': ['CD44', 'S100A4'],
#     'Treg': ['IL2RA', 'CTLA4'],
#     'CTL': ['GZMA', 'GZMH', 'NKG7', 'GNLY'],
#     'MAIT': ['SLC4A10'],
#     'gamma-delta': ['TRDC'],
#     'NKT': ['NCAM1', 'FCGR3A']
# }

sc.pl.dotplot(
    adata_T_clean,
    genes_ladder,
    groupby='cell_subtype',
    use_raw=True,
    standard_scale='var',  # 归一化让每个基因表达最强的簇显示为最深色
    cmap='Reds',
    figsize=(14, 6),
    #var_group_labels ,# 也可以通过字典形式传入来增加顶部分组注释
    show=True
)

In [142]:
# 1. 调整 Y 轴顺序，让 Naive 放在最上面
new_order = [
    'CD4 Naïve', 'CD8 Naïve', 
    'CD4 Memory', 'CD8 Memory', 
    'CD4 Treg', 
    'CD4 CTL', 'CD8 CTL', 
    'MAIT', 'γδT', 'NKT'
]
adata_T_clean.obs['cell_subtype'] = adata_T_clean.obs['cell_subtype'].cat.reorder_categories(new_order)

# 2. 调整 X 轴，把 Naive 基因提前，Treg 基因放在 Memory 之后 CTL 之前
genes_ladder = [
    'CD3D', 'TRAC', 'CD4', 'CD8A', 'CD8B',         # 基础分类
    'TRABD2A', 'MAL', 'CCR7', 'SELL', 'LEF1', 'TCF7', # Naive (对应顶部)
    'GPR183', 'CD44', 'S100A4',                    # Memory
    'IL2RA', 'CTLA4',                              # Treg
    'GZMA', 'GZMH', 'NKG7', 'GNLY', 'CX3CR1',      # CTL
    'SLC4A10', 'KLRB1',                            # MAIT
    'TRDC', 'TRDV1', 'TRDV2',                      # γδT
    'NCAM1', 'NCR1', 'KLRF1', 'KLRC2', 'KLRC3', 'TYROBP', 'FCGR3A' # NKT
]

In [143]:
sc.pl.dotplot(
    adata_T_clean,
    genes_ladder,
    groupby='cell_subtype',
    use_raw=True,
    standard_scale='var',  # 归一化让每个基因表达最强的簇显示为最深色
    cmap='Reds',
    figsize=(14, 6),
    #var_group_labels ,# 也可以通过字典形式传入来增加顶部分组注释
    show=True
)

In [145]:
sc.pl.umap(adata_T_clean, color=['cell_subtype']
           , legend_loc='on data'
          )

In [146]:
sc.pl.umap(adata_T_clean, color=['cell_subtype']
          # , legend_loc='on data'
          )